In [ ]:
import torch

import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR
import numpy as np
from tqdm.auto import tqdm
import pandas as pd
from src.pairwise_dataset import PairwiseDataset
import plotly.express as px
from sklearn.manifold import MDS, TSNE
from sklearn.decomposition import PCA
from umap import UMAP
from sklearn.linear_model import LinearRegression
from src.stimulis import Stimulis
from src.sentence_representations import SentenceRepresentations

torch.set_float32_matmul_precision("medium")

In [ ]:
stimulis = Stimulis(csv_path="datasets/relative_clause.csv")
df = stimulis.df
features = stimulis.features
repr = SentenceRepresentations(token_aggregation="first", layer=7)
Y = repr.compute_representations(stimulis.stimulis)

In [ ]:
proj = PCA(n_components=2).fit_transform(Y)

In [ ]:
df[["x", "y"]] = proj

In [ ]:
px.scatter(
    df,
    x="x",
    y="y",
    color="sentence_CLAUSE",
    hover_name=stimulis.stimulis,
    hover_data=features,
)

In [ ]:
proj = PCA(3).fit_transform(Y)

In [ ]:
proj = TSNE(2).fit_transform(Y.cpu().numpy())

In [ ]:
length = np.array([len(s.split()) for s in sentences])

In [ ]:
lr = LinearRegression()
lr.fit(Y, length)
# Y -= lr.predict(length)
# proj = PCA(2).fit_transform(Y)

In [ ]:
lr.coef_.shape

In [ ]:
proj = UMAP(2).fit_transform(Y.cpu().numpy())

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

lda = LinearDiscriminantAnalysis(n_components=2)
proj = lda.fit(Y.cpu(), df.sentence_length).transform(Y.cpu())

In [ ]:
# %%time
n_points = Y.shape[0]
n_components = 2
embeddings = torch.randn(
    n_points, n_components, device=device, requires_grad=True
)
optimizer = optim.AdamW([embeddings], lr=1)
scheduler = ReduceLROnPlateau(optimizer)
best_loss = torch.inf
patience = 0
losses = []
for i in (pbar := tqdm(range(10000), desc="Fitting MDS")):
    idx1, idx2, target_dist = dataset.sample(
        int(4096 * 64 * (1**i)), get_idx=True
    )
    optimizer.zero_grad(set_to_none=True)
    x = embeddings[idx1]
    y = embeddings[idx2]
    embedded_dist = (x - y).norm(dim=1, p=2)
    loss = ((embedded_dist - target_dist).pow(2)).mean()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    scheduler.step(loss.item())
    norm = embeddings.grad.detach().norm(p=2)
    pbar.set_postfix(
        loss=f"{loss:.4f}",
        lr=f"{optimizer.param_groups[0]['lr']:.1e}",
        norm=f"{norm:.4f}",
    )
    if loss < best_loss:
        best_loss = loss.item()
        patience = 0
    else:
        patience += 1

    if patience >= 20:
        break

In [ ]:
px.line(losses)

In [ ]:
px.scatter(
    embeddings.detach().cpu().numpy(),
    x=0,
    y=1,
    color=df.sentence_length,
    hover_name=sentences,
)

In [ ]:
px.scatter(proj, x=0, y=1, color=df.index, hover_name=sentences)

In [ ]:
df.columns

In [ ]:
proj = pca_after_removing_confounder(Y, lengths, 3)

In [ ]:
px.scatter(
    proj,
    x=0,
    y=1,
    color=df.sentence_GROUP + lengths.astype(str),
    hover_name=sentences,
)